# 11_adipose_liana_summary

Selected source cells for review and reconstruction.

Source: `Adipose_combined_samples_CellSegmentation/CellCellInteraction_Aging_1/Python LIANA+ Adipose Overall.ipynb`

All four Core/Neighbor/Distal exports. Figure styling trials excluded.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 3 (one-based); original execution count 2.

In [ ]:
import pandas as pd
import scanpy as sc
import scipy.io as sio
import liana as li

def load_export(outdir: str) -> sc.AnnData:
    X = sio.mmread(f"{outdir}/X.mtx").tocsr().T.tocsr()  # cells x genes
    genes = pd.read_csv(f"{outdir}/genes.txt", header=None)[0].tolist()
    cells = pd.read_csv(f"{outdir}/cells.txt", header=None)[0].tolist()
    obs = pd.read_csv(f"{outdir}/obs.csv").set_index("cell_id").loc[cells]
    adata = sc.AnnData(X=X)
    adata.var_names = genes
    adata.obs_names = cells
    adata.obs = obs
    return adata

# ---- 1) load ----
tag = "Adipose_Aged1"
outdir = "Adipose_Aged1_LIANA_export_SCT_CoreNeighborDistal"
adata = load_export(outdir)

# ---- 2) LIANA (mouse) ----
li.mt.rank_aggregate(
    adata,
    groupby="CCI_group",
    resource_name="mouseconsensus",
    use_raw=False,
    verbose=True
)
res = adata.uns["liana_res"].copy()

# ---- 3) mean差分（Neighbor>Distalのみ）----
df = res.copy()
df["lr"] = df["ligand_complex"].astype(str) + "→" + df["receptor_complex"].astype(str)

nb = df[(df["target"]=="Core") & (df["source"].str.startswith("Neighbor_"))].copy()
ds = df[(df["target"]=="Core") & (df["source"].str.startswith("Distal_"))].copy()

# 除外（必要なら）
nb = nb[~nb["source"].isin({"Neighbor_Erythroid cell","Neighbor_Unknown"})].copy()
ds = ds[~ds["source"].isin({"Distal_Erythroid cell","Distal_Unknown"})].copy()

nb["ct"] = nb["source"].str.replace("^Neighbor_", "", regex=True)
ds["ct"] = ds["source"].str.replace("^Distal_", "", regex=True)

nb_mean = nb.groupby(["ct","lr"], as_index=False)["lrscore"].mean().rename(columns={"lrscore":"mean_neighbor"})
ds_mean = ds.groupby(["ct","lr"], as_index=False)["lrscore"].mean().rename(columns={"lrscore":"mean_distal"})

m = nb_mean.merge(ds_mean, on=["ct","lr"], how="left")
m["mean_distal"] = m["mean_distal"].fillna(0.0)
m["delta"] = m["mean_neighbor"] - m["mean_distal"]

m_keep = m[m["delta"] > 0].copy()

m.shape, m_keep.shape, m_keep["ct"].value_counts().head(20)

Source cell 7 (one-based); original execution count 6.

In [ ]:
# ---- Aged1: keep results with unique names ----
aged1_res = res.copy()         # LIANA full result
aged1_m = m.copy()             # ct×LR (neighbor/distal/delta)
aged1_keep = m_keep.copy()     # delta>0 only (n=55)

# plot用の順序も保存しておく
aged1_ct_order = ct_order
aged1_lr_order = lr_order
aged1_dfp = dfp.copy()

Source cell 9 (one-based); original execution count 7.

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import scipy.io as sio
import liana as li
import matplotlib.pyplot as plt

def load_export(outdir: str) -> sc.AnnData:
    X = sio.mmread(f"{outdir}/X.mtx").tocsr().T.tocsr()  # cells x genes
    genes = pd.read_csv(f"{outdir}/genes.txt", header=None)[0].tolist()
    cells = pd.read_csv(f"{outdir}/cells.txt", header=None)[0].tolist()
    obs = pd.read_csv(f"{outdir}/obs.csv").set_index("cell_id").loc[cells]
    adata = sc.AnnData(X=X)
    adata.var_names = genes
    adata.obs_names = cells
    adata.obs = obs
    return adata

def neighbor_distal_keep(res_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = res_df.copy()
    df["lr"] = df["ligand_complex"].astype(str) + "→" + df["receptor_complex"].astype(str)

    nb = df[(df["target"]=="Core") & (df["source"].str.startswith("Neighbor_"))].copy()
    ds = df[(df["target"]=="Core") & (df["source"].str.startswith("Distal_"))].copy()

    # 除外（念のため）
    nb = nb[~nb["source"].isin({"Neighbor_Erythroid cell","Neighbor_Unknown"})].copy()
    ds = ds[~ds["source"].isin({"Distal_Erythroid cell","Distal_Unknown"})].copy()

    nb["ct"] = nb["source"].str.replace("^Neighbor_", "", regex=True)
    ds["ct"] = ds["source"].str.replace("^Distal_", "", regex=True)

    nb_mean = nb.groupby(["ct","lr"], as_index=False)["lrscore"].mean().rename(columns={"lrscore":"mean_neighbor"})
    ds_mean = ds.groupby(["ct","lr"], as_index=False)["lrscore"].mean().rename(columns={"lrscore":"mean_distal"})

    m = nb_mean.merge(ds_mean, on=["ct","lr"], how="left")
    m["mean_distal"] = m["mean_distal"].fillna(0.0)
    m["delta"] = m["mean_neighbor"] - m["mean_distal"]

    m_keep = m[m["delta"] > 0].copy()
    return df, m, m_keep

# ---- Aged2: load + LIANA ----
aged2_outdir = "Adipose_Aged2_LIANA_export_SCT_CoreNeighborDistal"
aged2_adata = load_export(aged2_outdir)

li.mt.rank_aggregate(
    aged2_adata,
    groupby="CCI_group",
    resource_name="mouseconsensus",
    use_raw=False,
    verbose=True
)
aged2_res = aged2_adata.uns["liana_res"].copy()

# ---- Aged2: mean差分でkeep ----
aged2_df, aged2_m, aged2_keep = neighbor_distal_keep(aged2_res)

aged2_m.shape, aged2_keep.shape, aged2_keep["ct"].value_counts()

# ---- Aged2: dotplot (all kept) ----
aged2_dfp = aged2_keep.copy()
s = aged2_dfp["mean_neighbor"].astype(float).values
aged2_dfp["size"] = (np.sqrt(s) - np.sqrt(s.min()) + 1e-6) * 250

aged2_ct_order = (aged2_dfp.groupby("ct")["mean_neighbor"].mean()
                  .sort_values(ascending=False).index.tolist())
aged2_lr_order = (aged2_dfp.groupby("lr")["mean_neighbor"].max()
                  .sort_values(ascending=False).index.tolist())

aged2_dfp["ct"] = pd.Categorical(aged2_dfp["ct"], categories=aged2_ct_order, ordered=True)
aged2_dfp["lr"] = pd.Categorical(aged2_dfp["lr"], categories=aged2_lr_order, ordered=True)

plt.figure(figsize=(3.3, max(6, 0.22*len(aged2_lr_order))))
plt.scatter(x=aged2_dfp["ct"], y=aged2_dfp["lr"], s=aged2_dfp["size"], alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.title("Aged2: Neighbor>Distal LR (dot size = mean lrscore Neighbor→Core)")
plt.xlabel("")
plt.ylabel("Ligand→Receptor")
plt.tight_layout()
plt.show()

Source cell 11 (one-based); original execution count 8.

In [ ]:
# ---- Young1: load + LIANA ----
young1_outdir = "Adipose_Young1_LIANA_export_SCT_CoreNeighborDistal"
young1_adata = load_export(young1_outdir)

li.mt.rank_aggregate(
    young1_adata,
    groupby="CCI_group",
    resource_name="mouseconsensus",
    use_raw=False,
    verbose=True
)
young1_res = young1_adata.uns["liana_res"].copy()

# ---- Young1: mean差分でkeep ----
young1_df, young1_m, young1_keep = neighbor_distal_keep(young1_res)

young1_m.shape, young1_keep.shape, young1_keep["ct"].value_counts()

# ---- Young1: dotplot（色変更） ----
young1_dfp = young1_keep.copy()
s = young1_dfp["mean_neighbor"].astype(float).values
young1_dfp["size"] = (np.sqrt(s) - np.sqrt(s.min()) + 1e-6) * 250

young1_ct_order = (young1_dfp.groupby("ct")["mean_neighbor"].mean()
                   .sort_values(ascending=False).index.tolist())
young1_lr_order = (young1_dfp.groupby("lr")["mean_neighbor"].max()
                   .sort_values(ascending=False).index.tolist())

young1_dfp["ct"] = pd.Categorical(young1_dfp["ct"], categories=young1_ct_order, ordered=True)
young1_dfp["lr"] = pd.Categorical(young1_dfp["lr"], categories=young1_lr_order, ordered=True)

plt.figure(figsize=(3.3, max(6, 0.22*len(young1_lr_order))))
plt.scatter(
    x=young1_dfp["ct"], y=young1_dfp["lr"],
    s=young1_dfp["size"],
    alpha=0.7,
    color="lightcoral"   # ←色を変更（好みで）
)
plt.xticks(rotation=45, ha="right")
plt.title("Young1: Neighbor>Distal LR (dot size = mean lrscore Neighbor→Core)")
plt.xlabel("")
plt.ylabel("Ligand→Receptor")
plt.tight_layout()
plt.show()

Source cell 13 (one-based); original execution count 9.

In [ ]:
# ---- Young2: load + LIANA ----
young2_outdir = "Adipose_Young2_LIANA_export_SCT_CoreNeighborDistal"
young2_adata = load_export(young2_outdir)

li.mt.rank_aggregate(
    young2_adata,
    groupby="CCI_group",
    resource_name="mouseconsensus",
    use_raw=False,
    verbose=True
)
young2_res = young2_adata.uns["liana_res"].copy()

# ---- Young2: mean差分でkeep ----
young2_df, young2_m, young2_keep = neighbor_distal_keep(young2_res)

young2_m.shape, young2_keep.shape, young2_keep["ct"].value_counts()

# ---- Young2: dotplot（Young1と同じ薄い赤系） ----
young2_dfp = young2_keep.copy()
s = young2_dfp["mean_neighbor"].astype(float).values
young2_dfp["size"] = (np.sqrt(s) - np.sqrt(s.min()) + 1e-6) * 250

young2_ct_order = (young2_dfp.groupby("ct")["mean_neighbor"].mean()
                   .sort_values(ascending=False).index.tolist())
young2_lr_order = (young2_dfp.groupby("lr")["mean_neighbor"].max()
                   .sort_values(ascending=False).index.tolist())

young2_dfp["ct"] = pd.Categorical(young2_dfp["ct"], categories=young2_ct_order, ordered=True)
young2_dfp["lr"] = pd.Categorical(young2_dfp["lr"], categories=young2_lr_order, ordered=True)

plt.figure(figsize=(3.3, max(6, 0.22*len(young2_lr_order))))
plt.scatter(
    x=young2_dfp["ct"], y=young2_dfp["lr"],
    s=young2_dfp["size"],
    alpha=0.7,
    color="lightcoral"
)
plt.xticks(rotation=45, ha="right")
plt.title("Young2: Neighbor>Distal LR (dot size = mean lrscore Neighbor→Core)")
plt.xlabel("")
plt.ylabel("Ligand→Receptor")
plt.tight_layout()
plt.show()

Source cell 15 (one-based); original execution count 10.

In [ ]:
import pandas as pd
import numpy as np

def combine_two(k1: pd.DataFrame, k2: pd.DataFrame, prefix: str) -> pd.DataFrame:
    # k1/k2: columns = ct, lr, mean_neighbor, mean_distal, delta
    a = k1[["ct","lr","mean_neighbor"]].rename(columns={"mean_neighbor": f"{prefix}1"})
    b = k2[["ct","lr","mean_neighbor"]].rename(columns={"mean_neighbor": f"{prefix}2"})
    m = a.merge(b, on=["ct","lr"], how="outer")

    m[f"{prefix}_present_n"] = (~m[f"{prefix}1"].isna()).astype(int) + (~m[f"{prefix}2"].isna()).astype(int)
    m[f"{prefix}_mean"] = m[[f"{prefix}1", f"{prefix}2"]].mean(axis=1, skipna=True)

    return m

aged_comb = combine_two(aged1_keep, aged2_keep, "Aged")
young_comb = combine_two(young1_keep, young2_keep, "Young")

aged_comb.head(), young_comb.head(), aged_comb.shape, young_comb.shape

Source cell 16 (one-based); original execution count 11.

In [ ]:
# 長い形式に変換
aged_long = aged_comb[["ct","lr","Aged_mean","Aged_present_n"]].copy()
aged_long["col"] = "Aged_" + aged_long["ct"]
aged_long = aged_long.rename(columns={"Aged_mean":"value", "Aged_present_n":"present_n"})

young_long = young_comb[["ct","lr","Young_mean","Young_present_n"]].copy()
young_long["col"] = "Young_" + young_long["ct"]
young_long = young_long.rename(columns={"Young_mean":"value", "Young_present_n":"present_n"})

plot_long = pd.concat([aged_long, young_long], ignore_index=True)

# valueがNAのところは点を描かないので落とす
plot_long = plot_long.dropna(subset=["value"]).copy()

plot_long.head(), plot_long["col"].nunique(), plot_long["lr"].nunique()

Source cell 23 (one-based); original execution count 32.

In [ ]:
import pandas as pd

# 1) Aged/Youngを (ct, lr) で結合（outer）
m = aged_comb[["ct","lr","Aged_mean","Aged_present_n"]].merge(
    young_comb[["ct","lr","Young_mean","Young_present_n"]],
    on=["ct","lr"], how="outer"
)

# 2) ligand/receptorに分解（lrは "Ligand→Receptor"）
m["ligand_complex"] = m["lr"].str.split("→", n=1).str[0]
m["receptor_complex"] = m["lr"].str.split("→", n=1).str[1]

# 3) 列順を整える
m_out = m[[
    "ct",
    "ligand_complex",
    "receptor_complex",
    "lr",
    "Aged_mean","Aged_present_n",
    "Young_mean","Young_present_n"
]].sort_values(["ct","ligand_complex","receptor_complex"])

# 4) 保存
m_out.to_csv("Adipose_Neighbor_to_Core_LR_table_AgedYoung.csv", index=False)

m_out.head()